# Lunar Tear Patcher (JP)

Patch a NieR Re\[in\]carnation **JP** APK and master data for use with a private server.

**How to use:**
1. Fill in the **Configuration** form below.
2. Run cells top-to-bottom, or use **Runtime → Run all**.
3. Patched files will download to your browser.

**Two modes:**
- **Drive mode** (default): Check **use_gdrive**, enter paths relative to `/content/drive/` (e.g. `MyDrive/original.apk`).
- **URL mode**: Uncheck **use_gdrive**, paste download URLs into `apk_source` / `masterdata_source`.

Each section is independent — skip what you don't need.


In [ ]:
#@title Configuration

#@markdown ### Source Mode
use_gdrive = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### File Sources
#@markdown With **use_gdrive** checked, enter Drive paths relative to `/content/drive/`
#@markdown (e.g. `MyDrive/original.apk`). Otherwise, enter download URLs.
apk_source = "MyDrive/NieR Re[in]carnation_3.7.1_JP.apk" #@param {type:"string"}
masterdata_source = "MyDrive/20240404193219.bin.e" #@param {type:"string"}

#@markdown ---
#@markdown ### Server Settings (JP)
grpc_addr = "ip:8003" #@param {type:"string"}
http_addr = "ip:8080" #@param {type:"string"}
auth_host = "ip:3000" #@param {type:"string"}

#@markdown ---
#@markdown ### Scripts Source (GitHub)
#@markdown Repository containing the patch scripts. If it is **private** (git clone asks for a
#@markdown username) or you hit GitHub rate limits, paste a Personal Access Token (repo scope)
#@markdown into `github_token`. Everything else stays the same.
scripts_repo = "https://github.com/lovelyzy7/lunar-scripts-jp.git" #@param {type:"string"}
scripts_branch = "main" #@param {type:"string"}
github_token = "" #@param {type:"string"}

import os

if use_gdrive:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive\n')
    for label, src in [('APK', apk_source), ('Master data', masterdata_source)]:
        if not src:
            print(f'  {label}: (not set, will skip)')
            continue
        path = os.path.join('/content/drive', src)
        if os.path.isfile(path):
            size = os.path.getsize(path)
            print(f'  {label}: {path} — OK ({size / 1024:.1f} KB)')
        else:
            print(f'  {label}: {path} — NOT FOUND')
else:
    print('URL mode — files will be downloaded during patching.')

for name, addr in [('grpc_addr', grpc_addr), ('http_addr', http_addr)]:
    if addr.startswith('ip:'):
        print(f'WARNING: {name} is still the placeholder "ip:..." — set your real server address.')


In [ ]:
#@title Install dependencies
import os, shutil, subprocess, sys

SCRIPTS_DIR = '/content/scripts-jp'

def run_step(cmd, label):
    """Run a shell command and raise a clear error instead of silently continuing."""
    print(f'>>> {cmd}')
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0:
        raise RuntimeError(f'{label} failed (exit code {rc}) — see the output above.')
    return rc

def sh_ok(cmd):
    """Run a shell command, return True on success (output shown)."""
    return subprocess.run(cmd, shell=True).returncode == 0

# ---------- 1) fetch the patch scripts ----------
def repo_url_with_token(url, token):
    if not token or not url.startswith('https://'):
        return url
    return 'https://' + token + '@' + url[len('https://'):]

patch_script = os.path.join(SCRIPTS_DIR, 'android/patch_apk_jp.py')

if os.path.isdir(os.path.join(SCRIPTS_DIR, '.git')):
    print(f'Found existing scripts at {SCRIPTS_DIR} — updating branch: {scripts_branch}')
    sh_ok(f'cd "{SCRIPTS_DIR}" && git fetch origin && git checkout {scripts_branch} && git pull origin {scripts_branch}')

if not os.path.isfile(patch_script):
    # a previous failed clone may have left an empty/broken directory behind
    if os.path.isdir(SCRIPTS_DIR):
        print(f'Removing incomplete scripts directory: {SCRIPTS_DIR}')
        shutil.rmtree(SCRIPTS_DIR, ignore_errors=True)

    clone_url = repo_url_with_token(scripts_repo, github_token.strip())
    print(f'Cloning {scripts_repo} (branch: {scripts_branch}) ...')
    if not sh_ok(f'git clone --depth 1 --branch {scripts_branch} "{clone_url}" "{SCRIPTS_DIR}"'):
        print('git clone failed — trying GitHub tarball download instead ...')
        slug = scripts_repo.rstrip('/').split('github.com/')[-1]
        if slug.endswith('.git'):
            slug = slug[:-4]
        api_url = f'https://api.github.com/repos/{slug}/tarball/{scripts_branch}'
        auth = f'-H "Authorization: token {github_token.strip()}"' if github_token.strip() else ''
        if sh_ok(f'curl -fL {auth} "{api_url}" -o /content/scripts.tar.gz'):
            shutil.rmtree('/content/scripts_extract', ignore_errors=True)
            os.makedirs('/content/scripts_extract', exist_ok=True)
            if sh_ok('tar -xzf /content/scripts.tar.gz -C /content/scripts_extract'):
                entries = os.listdir('/content/scripts_extract')
                if entries:
                    shutil.move(os.path.join('/content/scripts_extract', entries[0]), SCRIPTS_DIR)

# hard validation — never continue with missing scripts
REQUIRED = ['requirements.txt', 'android/patch_apk_jp.py', 'masterdata/patch_masterdata.py']
missing = [f for f in REQUIRED if not os.path.isfile(os.path.join(SCRIPTS_DIR, f))]
if missing:
    raise RuntimeError(
        'Patch scripts are missing from ' + SCRIPTS_DIR + ': ' + ', '.join(missing) + '\n'
        'Most likely the GitHub repo is private / deleted, the URL is wrong, or there is no '
        'network access.\n'
        'Fix: set `scripts_repo` (and `github_token` for a private repo) in the Configuration '
        'cell, then re-run this cell.'
    )

print('Patch scripts OK:')
for f in REQUIRED:
    print('  ' + os.path.join(SCRIPTS_DIR, f))

run_step(f'{sys.executable} -m pip install -q -r {SCRIPTS_DIR}/requirements.txt', 'pip install')

sh_ok('apt-get -qq update > /dev/null 2>&1')

# apktool — install from GitHub for a recent version
if not shutil.which('apktool'):
    run_step(
        'wget -q -O /usr/local/bin/apktool.jar '
        'https://github.com/iBotPeaches/Apktool/releases/download/v2.11.1/apktool_2.11.1.jar',
        'download apktool.jar')
    run_step(
        'wget -q -O /usr/local/bin/apktool '
        'https://raw.githubusercontent.com/iBotPeaches/Apktool/master/scripts/linux/apktool',
        'download apktool script')
    run_step('chmod +x /usr/local/bin/apktool', 'chmod apktool')

# zipalign + apksigner
if not shutil.which('zipalign') or not shutil.which('apksigner'):
    sh_ok('apt-get -qq install -y zipalign apksigner > /dev/null 2>&1')

tools_ok = True
for tool in ['apktool', 'zipalign', 'apksigner', 'keytool']:
    path = shutil.which(tool)
    if not path:
        tools_ok = False
    print(f'  {tool}: {path or "NOT FOUND"}')

if not tools_ok:
    raise RuntimeError('Some required tools were not found — see the list above.')

print('\nDone.')


In [ ]:
#@title Patch APK
import os, shutil, subprocess, sys, urllib.request
from google.colab import files

SCRIPTS_DIR = '/content/scripts-jp'
PATCH_SCRIPT = os.path.join(SCRIPTS_DIR, 'android/patch_apk_jp.py')

def run_step(cmd, label):
    """Run a shell command and raise a clear error instead of silently continuing."""
    print(f'>>> {cmd}')
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0:
        raise RuntimeError(f'{label} failed (exit code {rc}) — see the output above.')
    return rc

assert os.path.isfile(PATCH_SCRIPT), (
    f'Missing {PATCH_SCRIPT}. Run the "Install dependencies" cell first '
    '(and make sure it finished without errors).'
)
assert apk_source, 'Set apk_source in the Configuration cell first.'
assert grpc_addr, 'Set grpc_addr in the Configuration cell first.'
assert http_addr, 'Set http_addr in the Configuration cell first.'

if use_gdrive:
    apk_path = os.path.join('/content/drive', apk_source)
    assert os.path.isfile(apk_path), f'File not found: {apk_path}'
    shutil.copy2(apk_path, '/content/original.apk')
    size_mb = os.path.getsize('/content/original.apk') / 1024 / 1024
    print(f'Copied APK from Drive: {apk_path} ({size_mb:.1f} MB)')
else:
    print(f'Downloading APK from {apk_source} ...')
    urllib.request.urlretrieve(apk_source, '/content/original.apk')
    size_mb = os.path.getsize('/content/original.apk') / 1024 / 1024
    print(f'  Downloaded {size_mb:.1f} MB')

print('\nDecompiling ...')
run_step(f'apktool d /content/original.apk -o /content/patched -f', 'apktool decompile')

auth_flag = f'--auth-host "{auth_host}"' if auth_host else ''
print('\nPatching ...')
run_step(
    f'{sys.executable} "{PATCH_SCRIPT}" /content/patched '
    f'--grpc-addr "{grpc_addr}" --http-addr "{http_addr}" {auth_flag}',
    'patch_apk_jp')

print('\nRebuilding ...')
run_step('apktool b /content/patched -o /content/patched_unaligned.apk', 'apktool rebuild')

print('\nAligning ...')
run_step('zipalign -p -f 4 /content/patched_unaligned.apk /content/patched-jp.apk', 'zipalign')

if not os.path.exists('/content/debug.keystore'):
    print('\nGenerating debug keystore ...')
    run_step(
        'keytool -genkeypair -keystore /content/debug.keystore '
        '-alias key -storepass android -keypass android '
        '-keyalg RSA -keysize 2048 -validity 10000 -dname "CN=Debug" 2>/dev/null',
        'keytool')

print('\nSigning ...')
run_step('apksigner sign --ks /content/debug.keystore --ks-pass pass:android /content/patched-jp.apk',
         'apksigner')

assert os.path.isfile('/content/patched-jp.apk'), 'Patched APK was not produced.'
size_mb = os.path.getsize('/content/patched-jp.apk') / 1024 / 1024
print(f'\nDone! Patched APK: {size_mb:.1f} MB')

#files.download('/content/patched-jp.apk')


In [ ]:
#@title Save to Google Drive
import shutil
import os

#@markdown ### Patched APK
output_drive_path = "MyDrive/patched-jp.apk" #@param {type:"string"}

def save_to_drive(src, dst, label):
    if not dst:
        print(f"{label}: skipped (path not set)")
        return
    if not os.path.exists(src):
        print(f"{label}: not found at {src} — ensure the previous cell ran successfully.")
        return
    destination_path = os.path.join('/content/drive', dst)
    destination_dir = os.path.dirname(destination_path)
    if destination_dir and not os.path.exists(destination_dir):
        print(f"Creating directory: {destination_dir}")
        os.makedirs(destination_dir)
    try:
        shutil.copy2(src, destination_path)
        size_mb = os.path.getsize(destination_path) / 1024 / 1024
        print(f"{label}: successfully copied to Google Drive: {destination_path} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"{label}: error copying file to Google Drive: {e}")

if not use_gdrive:
    print("Google Drive is not configured. Please enable 'use_gdrive' in the Configuration cell if you wish to save to Drive.")
elif not os.path.isdir('/content/drive'):
    print("Google Drive is not mounted. Please ensure it's mounted correctly.")
else:
    save_to_drive('/content/patched-jp.apk', output_drive_path, 'Patched APK')


In [ ]:
#@title Patch Master Data
import os, shutil, subprocess, sys, urllib.request
from google.colab import files

SCRIPTS_DIR = '/content/scripts-jp'
MD_SCRIPT = os.path.join(SCRIPTS_DIR, 'masterdata/patch_masterdata.py')

def run_step(cmd, label):
    """Run a shell command and raise a clear error instead of silently continuing."""
    print(f'>>> {cmd}')
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0:
        raise RuntimeError(f'{label} failed (exit code {rc}) — see the output above.')
    return rc

assert os.path.isfile(MD_SCRIPT), (
    f'Missing {MD_SCRIPT}. Run the "Install dependencies" cell first '
    '(and make sure it finished without errors).'
)
assert masterdata_source, 'Set masterdata_source in the Configuration cell first.'

if use_gdrive:
    md_path = os.path.join('/content/drive', masterdata_source)
    assert os.path.isfile(md_path), f'File not found: {md_path}'
    shutil.copy2(md_path, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'Copied master data from Drive: {md_path} ({size_kb:.1f} KB)')
else:
    print(f'Downloading master data from {masterdata_source} ...')
    urllib.request.urlretrieve(masterdata_source, '/content/masterdata.bin.e')
    size_kb = os.path.getsize('/content/masterdata.bin.e') / 1024
    print(f'  Downloaded {size_kb:.1f} KB')

print('\nPatching ...')
run_step(
    f'{sys.executable} "{MD_SCRIPT}" --input /content/masterdata.bin.e --output /content/20240404193219.bin.e',
    'patch_masterdata')

assert os.path.isfile('/content/20240404193219.bin.e'), 'Patched master data was not produced.'
size_kb = os.path.getsize('/content/20240404193219.bin.e') / 1024
print(f'\nDone! Patched master data: {size_kb:.1f} KB')

#files.download('/content/20240404193219.bin.e')
